# 04 - Análisis exploratorio de datos (EDA)

Explora `demanda_gold` antes de entrenar cualquier modelo. Responde cinco preguntas:

| Sección | Pregunta |
|---|---|
| 1. Calidad | ¿Faltan datos? ¿Dónde y por qué? |
| 2. Objetivo | ¿Cómo se distribuye `pasajeros`? ¿Conviene transformarlo? |
| 3. Patrones en el tiempo | ¿Cómo cambia la demanda según la hora y el tipo de día? |
| 4. Líneas | ¿Qué tan distintas son las líneas entre sí? |
| 5. Calendario y clima | ¿Las clases, los feriados y la lluvia mueven la demanda? |
| 6. Relaciones entre variables | ¿Qué variables se parecen al objetivo? ¿Cuáles son redundantes entre sí? |

**Regla metodológica (Semana 2): el EDA se hace solo sobre `train`.** Mirar validación o test para
tomar decisiones (qué variables usar, qué transformación aplicar) es una fuga de información: el modelo
terminaría elegido con datos que se supone que no conoce. Val y test se usan recién en `05` a `07`.

In [0]:
%run ./00_setup

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

TIPOS_DIA = ["hábil", "sábado", "domingo", "turismo", "feriado"]
COLORES = {"hábil": "tab:blue", "sábado": "tab:orange", "domingo": "tab:green",
           "turismo": "tab:purple", "feriado": "tab:red"}


def con_tipo_dia(df):
    """Agrega 'tipo_dia'. El orden de los when importa: feriado le gana a todo, después Turismo."""
    return df.withColumn(
        "tipo_dia",
        F.when(F.col("es_no_laborable") | F.col("es_carnaval"), "feriado")
         .when(F.col("es_turismo"), "turismo")
         .when(F.col("dia_semana") == 7, "domingo")
         .when(F.col("dia_semana") == 6, "sábado")
         .otherwise("hábil"),
    )


gold = spark.table(config.TABLE_GOLD)
train = con_tipo_dia(gold.where(F.col("particion") == "train")).cache()

# Total del sistema por día: base de varias secciones
por_dia = (
    train.groupBy("fecha", "tipo_dia", "dia_semana", "es_feriado", "es_eleccion",
                  "clases_primaria", "clases_media", "nombre_dia_especial")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .toPandas()
    .sort_values("fecha")
)
por_dia["fecha"] = pd.to_datetime(por_dia["fecha"])
print(f"Días en train: {len(por_dia)}")

## 1. Calidad de los datos

Se espera:
- **Ceros**: muchos, por las horas de la noche y las líneas que no circulan todos los días.
- **Vacíos en rezagos**: las primeras 4 semanas de 2024, porque no hay días anteriores para mirar.
- **Vacíos en clima**: antes del 19/01/2024, cuando empiezan los pronósticos archivados.
- **Vacíos en `nombre_dia_especial`**: es normal, solo tiene valor en días especiales.

Cualquier otro vacío es una sorpresa y hay que investigarlo antes de seguir.

In [0]:
r = train.agg(
    F.count("*").alias("filas"),
    F.countDistinct("linea").alias("lineas"),
    F.min("fecha").alias("desde"),
    F.max("fecha").alias("hasta"),
    F.sum("pasajeros").alias("pasajeros"),
    F.avg((F.col("pasajeros") == 0).cast("int")).alias("prop_ceros"),
    F.avg(F.col("fila_completa").cast("int")).alias("prop_completas"),
    F.min(F.when(F.col("fila_completa"), F.col("fecha"))).alias("primera_completa"),
).first()

print(f"Filas:                {r['filas']:,}")
print(f"Líneas:               {r['lineas']}")
print(f"Fechas:               {r['desde']} a {r['hasta']}")
print(f"Pasajeros:            {r['pasajeros']:,}")
print(f"Horas con 0 pasajeros: {r['prop_ceros']:.1%}")
print(f"Filas completas:      {r['prop_completas']:.1%} (desde {r['primera_completa']})")

nulos = train.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in train.columns]).first().asDict()
nulos = pd.Series(nulos)
nulos = (nulos[nulos > 0] / r["filas"]).sort_values(ascending=False).rename("proporción vacía")
print("\nColumnas con vacíos:")
print(nulos.map("{:.2%}".format).to_string())

## 2. La variable objetivo: `pasajeros`

Estadística descriptiva (Semana 2): media, mediana, desvío, **asimetría** y **curtosis**.

- Si la media es mucho mayor que la mediana y la asimetría es grande y positiva, la distribución tiene
  una cola larga a la derecha: muchas horas con pocos pasajeros y pocas horas con muchísimos.
- En ese caso, el error cuadrático queda dominado por las horas pico. Una transformación
  $\log(1 + \text{pasajeros})$ achica esa cola (Semana 4). Si se usa, las métricas se reportan
  igual en pasajeros, deshaciendo la transformación.

In [0]:
d = train.agg(
    F.mean("pasajeros").alias("media"),
    F.stddev("pasajeros").alias("desvio"),
    F.skewness("pasajeros").alias("asimetria"),
    F.kurtosis("pasajeros").alias("curtosis"),
    F.max("pasajeros").alias("maximo"),
    F.expr("percentile_approx(pasajeros, array(0.25, 0.5, 0.75, 0.9, 0.99))").alias("q"),
).first()
q25, q50, q75, q90, q99 = d["q"]

print(f"Media:     {d['media']:8.1f}    Mediana: {q50}")
print(f"Desvío:    {d['desvio']:8.1f}    Q1 / Q3: {q25} / {q75}")
print(f"Asimetría: {d['asimetria']:8.2f}    P90 / P99: {q90} / {q99}")
print(f"Curtosis:  {d['curtosis']:8.2f}    Máximo: {d['maximo']}")

In [0]:
# Histogramas calculados en Spark (solo se traen los conteos)
ancho = max(1, int(np.ceil(q99 / 50)))
hist_lin = (train.where(F.col("pasajeros") <= q99)
            .groupBy((F.floor(F.col("pasajeros") / ancho) * ancho).alias("x")).count()
            .toPandas().sort_values("x"))
hist_log = (train.groupBy((F.floor(F.log1p("pasajeros") * 5) / 5).alias("x")).count()
            .toPandas().sort_values("x"))

fig, (a1, a2) = plt.subplots(1, 2)
a1.bar(hist_lin["x"], hist_lin["count"], width=ancho, align="edge")
a1.set(title=f"Pasajeros por hora (hasta el P99 = {q99})", xlabel="pasajeros", ylabel="filas")
a2.bar(hist_log["x"], hist_log["count"], width=0.2, align="edge")
a2.set(title="log(1 + pasajeros)", xlabel="log(1 + pasajeros)", ylabel="filas")
plt.tight_layout()
plt.show()

## 3. Patrones en el tiempo

### 3.1 Perfil horario según el tipo de día

Total del sistema en cada hora, promediado sobre los días de cada tipo. Qué mirar:
- ¿Hay dos picos (mañana y tarde) en días hábiles? ¿A qué hora?
- ¿Los sábados y domingos tienen otra forma, no solo otro nivel?

Si las curvas tienen **formas distintas**, un modelo lineal aditivo (hora + tipo de día) no alcanza:
hacen falta **interacciones** hora × tipo de día (Semana 4). Un árbol las encuentra solo.

In [0]:
perfil = (
    train.groupBy("fecha", "tipo_dia", "hora").agg(F.sum("pasajeros").alias("p"))
    .groupBy("tipo_dia", "hora").agg(F.avg("p").alias("pasajeros"), F.countDistinct("fecha").alias("dias"))
    .toPandas()
)

fig, ax = plt.subplots()
for t in TIPOS_DIA:
    s = perfil[perfil["tipo_dia"] == t].sort_values("hora")
    if len(s):
        ax.plot(s["hora"], s["pasajeros"], marker="o", ms=3, color=COLORES[t], label=f"{t} ({s['dias'].iloc[0]} días)")
ax.set(title="Perfil horario del sistema", xlabel="hora", ylabel="pasajeros promedio (todo el sistema)",
       xticks=range(0, 24, 2))
ax.legend()
plt.show()

### 3.2 Serie diaria

Total del sistema por día en train. Qué mirar:
- El ciclo semanal (caídas los fines de semana).
- Los días muy bajos: ¿todos están explicados por el calendario (feriados, Turismo, días sin servicio)?
  Si aparece un día bajo **sin explicación**, puede ser un paro: anotarlo para la variable de paros.
- Cambios de nivel: inicio de clases en marzo, vacaciones de julio.

In [0]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(por_dia["fecha"], por_dia["pasajeros"], color="lightgray", lw=1, zorder=1)
for t in TIPOS_DIA:
    s = por_dia[por_dia["tipo_dia"] == t]
    ax.scatter(s["fecha"], s["pasajeros"], s=10, color=COLORES[t], label=t, zorder=2)
ax.set(title="Pasajeros por día (todo el sistema)", ylabel="pasajeros")
ax.legend(ncol=5)
plt.show()

# Días hábiles anormalmente bajos (menos del 70 % de la mediana de su mes): candidatos a paro
habiles = por_dia[por_dia["tipo_dia"] == "hábil"].copy()
habiles["mediana_mes"] = habiles.groupby(habiles["fecha"].dt.month)["pasajeros"].transform("median")
habiles["ratio"] = habiles["pasajeros"] / habiles["mediana_mes"]
bajos = habiles[habiles["ratio"] < 0.7][["fecha", "pasajeros", "ratio", "nombre_dia_especial"]]
print("Días hábiles con menos del 70 % de la mediana de su mes:")
print(bajos.round({"ratio": 3}).to_string(index=False) if len(bajos) else "  ninguno")

## 4. Diferencias entre líneas

Qué mirar:
- **Concentración**: ¿qué parte de los pasajeros llevan las 10 o 20 líneas más grandes? Si es mucha,
  las métricas globales (RMSE, MAE) van a estar dominadas por esas líneas.
- **Media contra desvío**: si el desvío crece con la media (los puntos siguen una recta en escala
  log-log), el error de cada línea es proporcional a su tamaño. Es un argumento más para
  $\log(1 + \text{pasajeros})$.

In [0]:
lineas = (
    train.groupBy("linea", "tipo_linea")
    .agg(F.sum("pasajeros").alias("total"),
         F.avg("pasajeros").alias("media_hora"),
         F.stddev("pasajeros").alias("desvio_hora"),
         F.avg((F.col("pasajeros") == 0).cast("int")).alias("prop_ceros"))
    .toPandas().sort_values("total", ascending=False).reset_index(drop=True)
)
lineas["acumulado"] = lineas["total"].cumsum() / lineas["total"].sum()
for k in (10, 20, 50):
    if k <= len(lineas):
        print(f"Las {k} líneas más grandes llevan el {lineas['acumulado'].iloc[k - 1]:.1%} de los pasajeros")

fig, (a1, a2) = plt.subplots(1, 2)
a1.plot(np.arange(1, len(lineas) + 1), lineas["acumulado"], marker=".")
a1.set(title="Concentración de pasajeros", xlabel="cantidad de líneas (de mayor a menor)",
       ylabel="proporción acumulada", ylim=(0, 1.02))
a2.scatter(lineas["media_hora"], lineas["desvio_hora"], s=12)
a2.set(title="Media contra desvío por línea", xlabel="pasajeros por hora (media)",
       ylabel="desvío", xscale="log", yscale="log")
plt.tight_layout()
plt.show()

print("Las 5 líneas más chicas:")
print(lineas.tail(5)[["linea", "total", "media_hora", "prop_ceros"]].to_string(index=False))

## 5. Calendario y clima

### 5.1 Clases y feriados

Promedio de pasajeros por día según el tipo de día y si hay clases. Qué mirar:
- En días hábiles, ¿cuánto más viaja la gente cuando hay clases?
- ¿Los feriados laborables (el comercio abre) se parecen a un hábil o a un domingo?
- ¿Los domingos de elecciones se distinguen de un domingo común?

In [0]:
def tabla(df, por):
    return (df.groupby(por)["pasajeros"].agg(dias="count", promedio="mean")
            .round(0).astype({"promedio": "int64"}))

h = por_dia[por_dia["tipo_dia"] == "hábil"]
print("Días hábiles, según clases:")
print(tabla(h, ["clases_primaria", "clases_media"]).to_string(), "\n")

print("Días hábiles, feriado laborable o no:")
print(tabla(h, ["es_feriado"]).to_string(), "\n")

print("Domingos, con elecciones o sin:")
print(tabla(por_dia[por_dia["dia_semana"] == 7], ["es_eleccion"]).to_string())

### 5.2 Lluvia y temperatura pronosticadas

Comparar el total de días lluviosos y secos directamente mezcla otros efectos: en invierno llueve más,
y en invierno también hay clases. Para aislar el clima se usa un cociente:

$$\text{cociente del día} = \frac{\sum \text{pasajeros del día}}{\sum \text{mediana de 4 semanas del día}}$$

La mediana de las 4 semanas anteriores (misma línea, misma hora, mismo día de la semana) ya tiene en
cuenta la época del año y el día. Un cociente de 0,95 significa "un 5 % menos de lo habitual".
Se usan solo días hábiles con filas completas.

**Cuidado con los cambios de régimen.** La mediana mira 4 semanas hacia atrás. El primer mes de clases se
compara con semanas de vacaciones, así que su cociente sale alto aunque no haya llovido ni hecho calor; y
como marzo es cálido, aparecería un falso "efecto de la temperatura". Por eso se descartan los días cuyo
estado de clases (primaria y media) no es el mismo que en las 4 semanas de referencia.

Mirar siempre la columna `dias`: con pocas decenas de días, diferencias de 2 o 3 puntos en el cociente
pueden ser azar.

In [0]:
clima_dia = (
    con_tipo_dia(gold.where((F.col("particion") == "train") & F.col("fila_completa")))
    .where(F.col("tipo_dia") == "hábil")
    .groupBy("fecha")
    .agg((F.sum("pasajeros") / F.sum("pasajeros_mediana_4sem")).alias("cociente"),
         F.first("lluvia_pron_dia_mm").alias("lluvia_dia"),
         F.avg("temperatura_pron").alias("temp_media"))
    .toPandas()
)
clima_dia["fecha"] = pd.to_datetime(clima_dia["fecha"])

# Solo días con el mismo estado de clases que en d-7, d-14, d-21 y d-28
estado = por_dia.set_index("fecha")[["clases_primaria", "clases_media"]]


def regimen_estable(f):
    refs = [f - pd.Timedelta(days=k) for k in (7, 14, 21, 28)]
    return all(r in estado.index and (estado.loc[r] == estado.loc[f]).all() for r in refs)


antes = len(clima_dia)
clima_dia = clima_dia[clima_dia["fecha"].map(regimen_estable)]
print(f"Días hábiles completos: {antes}; con régimen de clases estable: {len(clima_dia)}\n")
clima_dia["lluvia"] = pd.cut(clima_dia["lluvia_dia"], [-0.01, 0.1, 5, 20, np.inf],
                             labels=["seco (< 0,1 mm)", "0,1 a 5 mm", "5 a 20 mm", "más de 20 mm"])
clima_dia["temperatura"] = pd.cut(clima_dia["temp_media"], [-np.inf, 10, 15, 20, 25, np.inf],
                                  labels=["< 10 °C", "10 a 15", "15 a 20", "20 a 25", "> 25 °C"])

for col in ("lluvia", "temperatura"):
    t = clima_dia.groupby(col, observed=False)["cociente"].agg(dias="count", cociente_medio="mean")
    print(f"Días hábiles por {col} pronosticada:")
    print(t.round(3).to_string(), "\n")

## 6. Relaciones entre variables

Correlación de Pearson sobre una muestra de filas completas de train. Dos lecturas (Semanas 2 y 3):

1. **Con el objetivo**: qué variables acompañan linealmente a `pasajeros`.
2. **Entre sí (multicolinealidad)**: variables casi iguales entre ellas hacen inestables los coeficientes
   de una regresión lineal. Por ejemplo, `pasajeros_prom_4sem` es exactamente el promedio de los rezagos de
   7, 14, 21 y 28 días: no pueden entrar todos juntos en una regresión lineal.

**Cuidado con la hora:** su correlación lineal con `pasajeros` va a dar baja, aunque la hora es de las
variables que más importan (ver 3.1). La correlación de Pearson solo mide relaciones en línea recta, y el
perfil horario tiene dos picos. Es el mismo mensaje del cuarteto de Anscombe: un número no alcanza.

In [0]:
NUMERICAS = [
    "pasajeros",
    "pasajeros_lag_1d", "pasajeros_lag_7d", "pasajeros_lag_14d", "pasajeros_lag_21d", "pasajeros_lag_28d",
    "pasajeros_prom_4sem", "pasajeros_mediana_4sem", "pasajeros_linea_dia_anterior",
    "temperatura_pron", "lluvia_pron_mm", "hora", "dia_semana",
]
completas = train.where(F.col("fila_completa"))
n = completas.count()
muestra = (completas.sample(fraction=min(1.0, 200_000 / max(n, 1)), seed=config.SEED)
           .select(*NUMERICAS).toPandas().astype(float))
corr = muestra.corr()

nombres = [c.replace("pasajeros_", "") for c in NUMERICAS]
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(nombres)), nombres, rotation=60, ha="right")
ax.set_yticks(range(len(nombres)), nombres)
ax.grid(False)
for i in range(len(nombres)):
    for j in range(len(nombres)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
ax.set_title(f"Correlación de Pearson (muestra de {len(muestra):,} filas completas de train)")
plt.tight_layout()
plt.show()

print("Correlación con pasajeros:")
print(corr["pasajeros"].drop("pasajeros").sort_values(ascending=False).round(3).to_string())

## Conclusiones

Resultados de la ejecución sobre train (01/01/2024 a 30/09/2024: 136 líneas, 274 días, 894.336 filas).

| Pregunta | Resultado | Decisión |
|---|---|---|
| ¿Hay vacíos inesperados? | No. Solo rezagos (enero), clima (antes del 19/01) y `nombre_dia_especial` | Modelar solo con `fila_completa` (89,8 % de las filas) |
| ¿La distribución de `pasajeros` es muy asimétrica? | Sí: media 251, mediana 67, asimetría 2,03; 24,4 % de horas en cero | Probar log(1 + pasajeros) en el modelo lineal; métricas siempre en pasajeros |
| ¿Los perfiles horarios cambian de forma según el tipo de día? | Sí. Hábil: pico 7-8, valle a las 10, meseta 12-18. Sábado: una loma (13 h). Domingo y feriado: meseta baja. Turismo ≈ sábado | Interacciones hora × tipo de día; crear `tipo_dia`; Turismo puede agruparse con sábado |
| ¿Hay días bajos sin explicación (posibles paros)? | No: los dos días hábiles bajos son feriados laborables (22/04 y 19/06) | Variable de paros con prioridad baja |
| ¿Qué tan concentrada está la demanda en pocas líneas? | 10 líneas = 24 %, 20 = 42 %, 50 = 80 % | Reportar el error global y por tamaño de línea |
| ¿El desvío crece con la media? | Sí, casi proporcional (recta en escala log-log) | Refuerza log(1 + pasajeros) |
| ¿Cuánto mueven las clases, los feriados y la lluvia? | Clases (ambos niveles) +25 %; feriado laborable −36 % (2 días); lluvia de 5 a 20 mm −15 % frente a un día seco. Temperatura: no concluyente (confundida con la vuelta de vacaciones). Elecciones: 1 solo día | Incluir clases, feriados y lluvia diaria; temperatura con cautela |
| ¿Qué variables son redundantes entre sí? | Rezagos semanales, promedio y mediana: correlaciones de 0,95 a 1,00 | Regresión lineal: `pasajeros_mediana_4sem` + `pasajeros_lag_1d` (como mucho `lag_7d`) |
| ¿Qué tan fuerte es la línea base? | `pasajeros_mediana_4sem` tiene correlación 0,972 con el objetivo | El R² contra la media no distingue modelos: comparar contra la mediana de 4 semanas |

**Consecuencia principal.** La demanda es muy repetitiva, así que "lo mismo que las últimas 4 semanas" acierta casi siempre.
El aporte de un modelo está en los días en que algo cambia respecto de esas semanas: feriados, inicio de clases,
lluvia fuerte, Turismo. En `07` hay que desglosar el error por tipo de día para mostrarlo.

**Limitaciones.** Pocos feriados laborables (2) y elecciones (1) en train; efecto de la temperatura no identificable;
las filas de un mismo día no son independientes (unos 245 días completos), así que los intervalos de las métricas se
calculan remuestreando días.

In [0]:
train.unpersist()